# Notebook 1: Data Preparation, Baseline, and Recovery (v2a)
## Hurricane Mobility Disruption — Largest Drop, Outflow Increase, Recovery Time

**Objective**: Process raw mobility data, establish SARIMA baselines, compute disruption/recovery metrics per county, export results + per-county plots.

**Flow types**: within-region, inflow, outflow  
**Hurricanes**: Milton (2024-10-09), Helene (2024-09-26)  
**SARIMA**: AR(1) with exogenous dummies, order=(1,0,0), seasonal=(0,0,0,0)  

**Metrics computed per county:**
- **Largest drop** (within, inflow): min % deviation from baseline during landing week
- **Outflow increase** (outflow): max % deviation above baseline during week before + landing week
- **Recovery time** (within, inflow): Theil-Sen trend-based recovery days from landing

**Exports per hurricane:**
- `baseline_{within,inflow,outflow}.csv` — SARIMA predictions
- `largest_drop_{within,inflow}.csv` — peak disruption
- `outflow_increase.csv` — evacuation surge
- `recovery_{within,inflow}.csv` — trend-based recovery time
- `figures/` — per-county baseline validation + recovery plots

In [1]:
import pandas as pd
import numpy as np
import h5py
import os
import sys
import datetime as dt
import warnings
from importlib import reload

from scipy.stats import theilslopes
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

warnings.filterwarnings("ignore", category=FutureWarning)

In [2]:
folder_path = "./../../hurricane_oct/"

sys.path.append(folder_path)
sys.path.append(os.path.join(folder_path, "mobility_function"))

from mobility_function import analysis as ma
ma = reload(ma)

%run ./recovery_function_v2.py

In [3]:
# ── Configuration ──

HURRICANES = {
    "milton": {
        "landing_date": pd.Timestamp("2024-10-09"),
        "county_file": "../results/milton/counties_geoid_cut_50.txt",
        "output_dir": "../results/milton/",
    },
    "helene": {
        "landing_date": pd.Timestamp("2024-09-26"),
        "county_file": "../results/helene/counties_geoid_cut_50.txt",
        "output_dir": "../results/helene/",
    },
}

# SARIMA specification
ARIMA_ORDER = (1, 0, 0)
SEASONAL_ORDER = (0, 0, 0, 0)

# Flow types
FLOW_TYPES = ["within", "inflow", "outflow"]

# Trend-based recovery parameters
SMOOTH_WINDOW = 3
TROUGH_SEARCH_DAYS = 7


def mondays_str(year, start_month=7, end_month=10):
    """Generate Monday date strings for weekly H5 files."""
    start = dt.date(year, start_month, 28)
    end = dt.date(year, end_month, 31)
    days_ahead = (0 - start.weekday()) % 7
    cur = start + dt.timedelta(days=days_ahead)
    out = []
    while cur <= end:
        out.append(cur.strftime("%Y%m%d"))
        cur += dt.timedelta(days=7)
    return out


mondays_2023 = mondays_str(2023)
mondays_2024 = mondays_str(2024)
all_mondays = mondays_2023 + mondays_2024

# Build date index
dates_2023 = pd.date_range(start="2023-07-31", periods=len(mondays_2023) * 7, freq="D")
dates_2024 = pd.date_range(start="2024-07-29", periods=len(mondays_2024) * 7, freq="D")
dates_all = dates_2023.union(dates_2024)

print(f"Mondays 2023: {len(mondays_2023)}, Mondays 2024: {len(mondays_2024)}")
print(f"Total days in date index: {len(dates_all)}")

Mondays 2023: 14, Mondays 2024: 14
Total days in date index: 196


In [4]:
geo_idx = pd.read_csv("geoid_idx_names.csv")
print(f"Geoid mapping: {len(geo_idx)} counties")
display(geo_idx.head())

Geoid mapping: 3144 counties


,GEOID,county_idx,NAME
0,12115,0,Sarasota
1,12081,1,Manatee
2,12027,2,DeSoto
3,12015,3,Charlotte
4,12049,4,Hardee


## 1. Helper Functions

In [5]:
def load_county_flows(selected_idx):
    """
    Load H5 mobility data and decompose into within-region, inflow, and outflow
    per county using ma.region_mobility().

    Returns
    -------
    flows : dict
        {"within": array (days, n_counties),
         "inflow": array (days, n_counties),
         "outflow": array (days, n_counties)}
        All 17 categories summed together.
    """
    # Process first week
    M_0 = ma.h5py_to_4d_array(folder_path + f"data/mobility/M_raw_{all_mondays[0]}.h5")
    W_0, O_0, I_0 = ma.region_mobility(M_0, selected_idx)
    W_ts = W_0
    O_ts = O_0
    I_ts = I_0

    for j, date_str in enumerate(all_mondays[1:], 1):
        print(f"  Loading {date_str} ({j+1}/{len(all_mondays)})...", end="")
        M_i = ma.h5py_to_4d_array(folder_path + f"data/mobility/M_raw_{date_str}.h5")
        W_i, O_i, I_i = ma.region_mobility(M_i, selected_idx)
        W_ts = np.concatenate((W_ts, W_i), axis=0)
        O_ts = np.concatenate((O_ts, O_i), axis=0)
        I_ts = np.concatenate((I_ts, I_i), axis=0)
        print(" done")

    print(f"\nW_ts (within)  shape: {W_ts.shape}")
    print(f"O_ts (outflow) shape: {O_ts.shape}")
    print(f"I_ts (inflow)  shape: {I_ts.shape}")

    # Sum all 17 categories -> (days, n_counties)
    W_county = W_ts.sum(axis=1)
    O_county = O_ts.sum(axis=1)
    I_county = I_ts.sum(axis=1)

    print(f"After category sum — within: {W_county.shape}, "
          f"outflow: {O_county.shape}, inflow: {I_county.shape}")

    return {"within": W_county, "inflow": I_county, "outflow": O_county}

In [6]:
def compute_baseline_and_drop(county_flow, dates_all, county_geoid_list,
                              hrc_config, landing_date):
    """
    For each county (within/inflow):
    1. Fit SARIMA baseline
    2. Generate predictions with CI
    3. Compute largest DROP within [landing, landing+6d]

    Returns
    -------
    drop_df : DataFrame [GEOID, largest_drop, trough_date]
    baseline_records : dict {GEOID: df_rec DataFrame}
    failed_counties : list of (GEOID, error_message)
    """
    train_end = hrc_config["train_2024_end"]
    fc_start = hrc_config["forecast_start"]
    fc_end = hrc_config["forecast_end"]
    landing_week_end = landing_date + pd.Timedelta(days=6)

    results = []
    baseline_records = {}
    failed_counties = []
    n_counties = len(county_geoid_list)

    for i, geoid in enumerate(county_geoid_list):
        if (i + 1) % 25 == 0 or i == 0:
            print(f"  Processing county {i+1}/{n_counties}: GEOID {geoid}")

        try:
            y_log, y, X = prepare_time_series_with_exog(county_flow[:, i], dates_all)
            res, _, _ = fit_arimax_model(
                y_log, X,
                order=ARIMA_ORDER, seasonal_order=SEASONAL_ORDER,
                train_2024_end=train_end,
            )
            df_rec, forecast_idx = get_predictions_and_ci(
                res, X, y,
                forecast_start=fc_start, forecast_end=fc_end,
            )

            # Relative diff
            eps = 1e-12
            denom = df_rec["y_pred"].replace(0, np.nan) + eps
            df_rec["relative_diff"] = (df_rec["y_true"] - df_rec["y_pred"]) / denom * 100

            # Largest drop within landing week
            week_mask = (df_rec.index >= landing_date) & (df_rec.index <= landing_week_end)
            week_data = df_rec.loc[week_mask, "relative_diff"]
            if week_data.empty:
                raise ValueError("No data in landing week window")

            largest_drop = week_data.min()
            trough_date = week_data.idxmin()
            baseline_records[geoid] = df_rec.copy()

        except Exception as e:
            print(f"    SARIMA failed for GEOID {geoid}: {e}")
            largest_drop = np.nan
            trough_date = None
            failed_counties.append((geoid, str(e)))

        results.append({
            "GEOID": int(geoid),
            "largest_drop": largest_drop,
            "trough_date": trough_date,
        })

    df = pd.DataFrame(results)
    n_valid = df["largest_drop"].notna().sum()
    print(f"\nCompleted: {n_valid}/{n_counties} counties with valid largest_drop")
    return df, baseline_records, failed_counties

In [7]:
def compute_outflow_increase(county_flow, dates_all, county_geoid_list,
                              hrc_config, landing_date):
    """
    For each county (outflow):
    1. Fit SARIMA baseline
    2. Compute largest POSITIVE deviation within [landing-7d, landing+6d]
       (captures pre-landfall evacuation + landing-week surge)

    Returns
    -------
    increase_df : DataFrame [GEOID, largest_increase, peak_date]
    baseline_records : dict {GEOID: df_rec DataFrame}
    failed_counties : list of (GEOID, error_message)
    """
    train_end = hrc_config["train_2024_end"]
    fc_start = hrc_config["forecast_start"]
    fc_end = hrc_config["forecast_end"]

    # 2-week window: week before landing + landing week
    window_start = landing_date - pd.Timedelta(days=7)
    window_end = landing_date + pd.Timedelta(days=6)

    results = []
    baseline_records = {}
    failed_counties = []
    n_counties = len(county_geoid_list)

    for i, geoid in enumerate(county_geoid_list):
        if (i + 1) % 25 == 0 or i == 0:
            print(f"  Processing county {i+1}/{n_counties}: GEOID {geoid}")

        try:
            y_log, y, X = prepare_time_series_with_exog(county_flow[:, i], dates_all)
            res, _, _ = fit_arimax_model(
                y_log, X,
                order=ARIMA_ORDER, seasonal_order=SEASONAL_ORDER,
                train_2024_end=train_end,
            )
            df_rec, forecast_idx = get_predictions_and_ci(
                res, X, y,
                forecast_start=fc_start, forecast_end=fc_end,
            )

            # Relative diff
            eps = 1e-12
            denom = df_rec["y_pred"].replace(0, np.nan) + eps
            df_rec["relative_diff"] = (df_rec["y_true"] - df_rec["y_pred"]) / denom * 100

            # Largest INCREASE within 2-week window
            win_mask = (df_rec.index >= window_start) & (df_rec.index <= window_end)
            win_data = df_rec.loc[win_mask, "relative_diff"]
            if win_data.empty:
                raise ValueError("No data in outflow search window")

            largest_increase = win_data.max()
            peak_date = win_data.idxmax()
            baseline_records[geoid] = df_rec.copy()

        except Exception as e:
            print(f"    SARIMA failed for GEOID {geoid}: {e}")
            largest_increase = np.nan
            peak_date = None
            failed_counties.append((geoid, str(e)))

        results.append({
            "GEOID": int(geoid),
            "largest_increase": largest_increase,
            "peak_date": peak_date,
        })

    df = pd.DataFrame(results)
    n_valid = df["largest_increase"].notna().sum()
    print(f"\nCompleted: {n_valid}/{n_counties} counties with valid outflow increase")
    return df, baseline_records, failed_counties

### Trend-Based Recovery Functions

Adapted from `trend_based_recovery_region_v2.ipynb` (cell-11).  
Method: smooth relative deviation → find trough → extract monotonic recovery segment → Theil-Sen robust slope → recovery time from landing.

In [8]:
def trend_based_recovery(df_rec, landing_date, y_col="y_true", yhat_col="y_pred",
                         smooth_window=3, trough_search_days=7):
    """
    Trend-based recovery time with robust Theil-Sen estimation.
    Recovery time is measured from the landing date.

    Steps:
    1. Compute relative deviation rd = (y_true - y_pred) / y_pred
    2. Smooth with centred moving average
    3. Find trough within trough_search_days after landing
    4. Extract monotonic recovery segment (non-decreasing from trough)
    5. Fit Theil-Sen slope on monotonic segment
    6. Recovery time = (trough - landing) + (-intercept / slope)
    """
    y = df_rec[y_col].astype(float)
    yhat = df_rec[yhat_col].astype(float)

    eps = 1e-12
    denom = yhat.replace(0, np.nan) + eps
    relative_diff = (y - yhat) / denom

    # Smooth
    rd_smooth = relative_diff.rolling(window=smooth_window, center=True, min_periods=1).mean()

    # Trough within [landing, landing + trough_search_days]
    trough_end = landing_date + pd.Timedelta(days=trough_search_days)
    search_window = rd_smooth.loc[
        (rd_smooth.index >= landing_date) & (rd_smooth.index <= trough_end)
    ]
    if search_window.empty or search_window.min() >= 0:
        return dict(
            trough_date=None, recovery_days=None, recovery_date=None,
            intercept=None, slope=None,
            reason=f"No negative deviation found within {trough_search_days}d of landing.",
            relative_diff=relative_diff, rd_smooth=rd_smooth,
        )

    trough_date = search_window.idxmin()

    # Monotonic recovery segment
    post_trough = rd_smooth.loc[rd_smooth.index >= trough_date]
    vals = post_trough.values
    mono_end = 1
    for i in range(1, len(vals)):
        if vals[i] >= vals[i - 1] - 1e-15:
            mono_end = i + 1
        else:
            break
    mono_segment = post_trough.iloc[:mono_end]

    if len(mono_segment) < 2:
        return dict(
            trough_date=trough_date, recovery_days=None, recovery_date=None,
            intercept=None, slope=None,
            reason="Monotonic recovery segment too short for fitting.",
            relative_diff=relative_diff, rd_smooth=rd_smooth,
        )

    # Theil-Sen slope
    t = np.arange(len(mono_segment), dtype=float)
    slope, intercept, lo_slope, hi_slope = theilslopes(mono_segment.values, t)

    if slope <= 0:
        return dict(
            trough_date=trough_date, recovery_days=None, recovery_date=None,
            intercept=float(intercept), slope=float(slope),
            reason="Theil-Sen slope <= 0; no upward recovery trend.",
            relative_diff=relative_diff, rd_smooth=rd_smooth,
            mono_segment=mono_segment,
        )

    # Recovery time from landing
    tau_from_trough = -intercept / slope
    recovery_date = trough_date + pd.to_timedelta(tau_from_trough, unit="D")
    days_landing_to_trough = (trough_date - landing_date).days
    recovery_days = days_landing_to_trough + tau_from_trough

    return dict(
        trough_date=trough_date,
        days_landing_to_trough=days_landing_to_trough,
        tau_from_trough=float(tau_from_trough),
        recovery_days=float(recovery_days),
        recovery_date=recovery_date,
        intercept=float(intercept),
        slope=float(slope),
        slope_ci=(float(lo_slope), float(hi_slope)),
        relative_diff=relative_diff,
        rd_smooth=rd_smooth,
        mono_segment=mono_segment,
        post_trough=post_trough,
    )

In [9]:
def compute_county_recovery(baselines, county_geoid_list, landing_date):
    """
    Compute trend-based recovery for each county using pre-computed baselines.
    Reuses SARIMA results from compute_baseline_and_drop() — no duplicate fitting.

    Returns
    -------
    recovery_df : DataFrame [GEOID, recovery_days, trough_date, recovery_date,
                             slope, intercept, mono_segment_days]
    recovery_results : dict {GEOID: trend_based_recovery result dict}
    """
    rows = []
    recovery_results = {}
    n = len(county_geoid_list)

    for i, geoid in enumerate(county_geoid_list):
        if (i + 1) % 50 == 0 or i == 0:
            print(f"  Recovery {i+1}/{n}: GEOID {geoid}")

        if geoid not in baselines:
            rows.append({"GEOID": int(geoid), "recovery_days": np.nan})
            continue

        df_rec = baselines[geoid]
        result = trend_based_recovery(
            df_rec, landing_date,
            smooth_window=SMOOTH_WINDOW,
            trough_search_days=TROUGH_SEARCH_DAYS,
        )
        recovery_results[geoid] = result

        mono = result.get("mono_segment")
        rows.append({
            "GEOID": int(geoid),
            "recovery_days": result.get("recovery_days"),
            "trough_date": result.get("trough_date"),
            "recovery_date": result.get("recovery_date"),
            "slope": result.get("slope"),
            "intercept": result.get("intercept"),
            "mono_segment_days": len(mono) if mono is not None else None,
        })

    df = pd.DataFrame(rows)
    n_valid = df["recovery_days"].notna().sum()
    print(f"\nRecovery completed: {n_valid}/{n} counties with valid recovery time")
    return df, recovery_results

### Plotting and Export Functions

In [10]:
def export_all_baseline_plots(baselines, hrc_name, flow_type, landing_date,
                               output_dir):
    """
    Save one baseline validation plot per county.
    Path: {output_dir}/figures/baseline_{flow_type}_GEOID_{geoid}.png
    """
    fig_dir = os.path.join(output_dir, "figures")
    os.makedirs(fig_dir, exist_ok=True)

    geoid_list = list(baselines.keys())
    print(f"Exporting {len(geoid_list)} baseline plots for {hrc_name}/{flow_type}...")

    for geoid in geoid_list:
        df = baselines[geoid]
        fig, ax = plt.subplots(figsize=(12, 4))
        ax.plot(df.index, df["y_true"], label="Observed", color="black", linewidth=1.5)
        ax.plot(df.index, df["y_pred"], label="Predicted (baseline)",
                color="red", linewidth=1.5)
        ax.fill_between(df.index, df["ci_lower"], df["ci_upper"],
                        color="red", alpha=0.15, label="95% CI")
        ax.axvline(landing_date, color="blue", linestyle="--", linewidth=1.5,
                   label="Landing date")
        ax.set_title(f"{hrc_name.capitalize()} — GEOID {geoid} — {flow_type}")
        ax.set_ylabel("Mobility (flow)")
        ax.legend(fontsize=8)
        ax.grid(alpha=0.3)
        plt.tight_layout()

        save_path = os.path.join(fig_dir, f"baseline_{flow_type}_GEOID_{geoid}.png")
        fig.savefig(save_path, dpi=150, bbox_inches="tight")
        plt.close(fig)

    print(f"  Done: {len(geoid_list)} plots saved to {fig_dir}/")

In [11]:
def plot_trend_recovery_county(result, forecast_idx, geoid, flow_type,
                                hrc_name, landing_date, save_path):
    """
    Single-county recovery plot: raw + smoothed relative diff, trough,
    monotonic segment, Theil-Sen fit, recovery annotation.
    Adapted from trend_based_recovery_region_v2.ipynb cell-13.
    """
    relative_diff = result["relative_diff"]
    rd_smooth = result["rd_smooth"]
    trough_date = result.get("trough_date")
    recovery_date = result.get("recovery_date")
    recovery_days = result.get("recovery_days")

    fig, ax = plt.subplots(figsize=(12, 5))

    # Raw relative difference (thin, faded)
    ax.plot(forecast_idx, relative_diff.reindex(forecast_idx).values * 100,
            color="black", linewidth=0.8, alpha=0.35, label="Raw relative diff (%)")

    # Smoothed (bold)
    ax.plot(forecast_idx, rd_smooth.reindex(forecast_idx).values * 100,
            color="black", linewidth=2, label="Smoothed (MA)")

    ax.axhline(0, color="gray", linestyle="--", linewidth=1, alpha=0.7)

    # Landing date
    ax.axvline(landing_date, color="blue", linestyle="--", linewidth=1.5,
               label=f"{landing_date.date()} landing")

    # Trough
    if trough_date is not None:
        ax.axvline(trough_date, color="orange", linestyle="--", linewidth=2,
                   label=f"Trough ({trough_date.date()})")

    # Monotonic segment
    mono = result.get("mono_segment")
    if mono is not None and len(mono) >= 2:
        ax.plot(mono.index, mono.values * 100,
                color="#2ca02c", linewidth=3, alpha=0.5,
                label=f"Monotonic segment ({len(mono)}d)")

    # Theil-Sen fit line
    post_trough = result.get("post_trough")
    if post_trough is not None and result.get("intercept") is not None:
        a = result["intercept"]
        b = result["slope"]
        t_fit = np.arange(len(post_trough))
        r_fit = a + b * t_fit
        ax.plot(post_trough.index, r_fit * 100,
                color="green", linewidth=2, linestyle="-",
                label=f"Theil-Sen (slope={b*100:.2f}%/d)")

    # Recovery date
    if recovery_date is not None and recovery_days is not None:
        ax.axvline(recovery_date, color="green", linestyle=":", linewidth=2,
                   label=f"Recovery ({recovery_days:.1f}d from landing)")
        ax.text(recovery_date, ax.get_ylim()[1] * 0.85,
                f"  {recovery_days:.1f}d", color="green",
                fontsize=10, va="top", ha="left", rotation=90)

    ax.set_title(f"Recovery — GEOID {geoid} / {flow_type.capitalize()} "
                 f"({hrc_name.capitalize()})")
    ax.set_xlabel("Date")
    ax.set_ylabel("Relative Difference (%)")
    ax.legend(loc="lower right", fontsize=8)
    ax.grid(True, alpha=0.3)
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%m-%d"))
    ax.xaxis.set_major_locator(mdates.WeekdayLocator(byweekday=mdates.MO))
    fig.autofmt_xdate(rotation=45)
    fig.tight_layout()

    fig.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.close(fig)

In [12]:
def export_all_recovery_plots(recovery_results, forecast_idx, county_geoid_list,
                               flow_type, hrc_name, landing_date, output_dir):
    """
    Save one recovery plot per county.
    Path: {output_dir}/figures/recovery_{flow_type}_GEOID_{geoid}.png
    """
    fig_dir = os.path.join(output_dir, "figures")
    os.makedirs(fig_dir, exist_ok=True)

    count = 0
    for geoid in county_geoid_list:
        if geoid not in recovery_results:
            continue
        result = recovery_results[geoid]
        if result.get("rd_smooth") is None:
            continue

        save_path = os.path.join(fig_dir, f"recovery_{flow_type}_GEOID_{geoid}.png")
        plot_trend_recovery_county(
            result, forecast_idx, geoid, flow_type,
            hrc_name, landing_date, save_path
        )
        count += 1

    print(f"  Exported {count} recovery plots for {hrc_name}/{flow_type}")

In [13]:
def export_baselines(baselines, drop_df, output_dir, flow_type):
    """
    Export baseline predictions and drop/increase results.
    """
    os.makedirs(output_dir, exist_ok=True)

    # Baseline: long-format
    baseline_rows = []
    for geoid, df in baselines.items():
        df_copy = df.copy()
        df_copy["GEOID"] = geoid
        df_copy["date"] = df_copy.index
        baseline_rows.append(df_copy)

    if baseline_rows:
        baseline_all = pd.concat(baseline_rows, ignore_index=True)
        baseline_path = os.path.join(output_dir, f"baseline_{flow_type}.csv")
        baseline_all.to_csv(baseline_path, index=False)
        print(f"Exported baseline to {baseline_path} ({len(baseline_all)} rows)")

    # Drop/increase results
    result_path = os.path.join(output_dir,
        f"largest_drop_{flow_type}.csv" if "largest_drop" in drop_df.columns
        else f"outflow_increase.csv"
    )
    drop_df.to_csv(result_path, index=False)
    print(f"Exported to {result_path} ({len(drop_df)} rows)")


def export_recovery(recovery_df, output_dir, flow_type):
    """Export recovery time results."""
    os.makedirs(output_dir, exist_ok=True)
    path = os.path.join(output_dir, f"recovery_{flow_type}.csv")
    recovery_df.to_csv(path, index=False)
    print(f"Exported recovery to {path} ({len(recovery_df)} rows)")

---
## 2. Hurricane Milton

In [14]:
# ── Load Milton county list ──
hrc_name = "milton"
hrc_cfg = HURRICANES[hrc_name]
landing_date = hrc_cfg["landing_date"]
output_dir = hrc_cfg["output_dir"]

with open(hrc_cfg["county_file"], "r") as f:
    milton_county_list = [int(line.strip()) for line in f if line.strip()]

milton_geo = geo_idx[geo_idx["GEOID"].isin(milton_county_list)].sort_values("county_idx")
milton_selected_idx = milton_geo["county_idx"].values
milton_geoids = milton_geo["GEOID"].tolist()

train_2024_end = (landing_date - pd.Timedelta(days=7)).strftime("%Y-%m-%d")
forecast_start = (landing_date - pd.Timedelta(days=6)).strftime("%Y-%m-%d")
forecast_end = "2024-10-31"

milton_hrc_config = {
    "train_2024_end": train_2024_end,
    "forecast_start": forecast_start,
    "forecast_end": forecast_end,
}
milton_forecast_idx = pd.date_range(forecast_start, forecast_end, freq="D")

print(f"Milton: {len(milton_geoids)} affected counties")
print(f"Landing date: {landing_date.date()}")
print(f"Train end: {train_2024_end}, Forecast: {forecast_start} -> {forecast_end}")

Milton: 21 affected counties
Landing date: 2024-10-09
Train end: 2024-10-02, Forecast: 2024-10-03 -> 2024-10-31


In [15]:
# ── Load Milton mobility data (all 3 flow types) ──
print("Loading Milton county-level flow data...")
milton_flows = load_county_flows(milton_selected_idx)

Loading Milton county-level flow data...
  Loading 20230807 (2/28)... done
  Loading 20230814 (3/28)... done
  Loading 20230821 (4/28)... done
  Loading 20230828 (5/28)... done
  Loading 20230904 (6/28)... done
  Loading 20230911 (7/28)... done
  Loading 20230918 (8/28)... done
  Loading 20230925 (9/28)... done
  Loading 20231002 (10/28)... done
  Loading 20231009 (11/28)... done
  Loading 20231016 (12/28)... done
  Loading 20231023 (13/28)... done
  Loading 20231030 (14/28)... done
  Loading 20240729 (15/28)... done
  Loading 20240805 (16/28)... done
  Loading 20240812 (17/28)... done
  Loading 20240819 (18/28)... done
  Loading 20240826 (19/28)... done
  Loading 20240902 (20/28)... done
  Loading 20240909 (21/28)... done
  Loading 20240916 (22/28)... done
  Loading 20240923 (23/28)... done
  Loading 20240930 (24/28)... done
  Loading 20241007 (25/28)... done
  Loading 20241014 (26/28)... done
  Loading 20241021 (27/28)... done
  Loading 20241028 (28/28)... done

W_ts (within)  shape:

### 2.1 Compute Largest Drop (within + inflow)

In [16]:
milton_drop_dfs = {}
milton_baselines = {}
milton_failures = {}

for ft in ["within", "inflow"]:
    print(f"\n--- Milton: Largest drop for {ft.upper()} ---")
    drop_df, baselines, failures = compute_baseline_and_drop(
        milton_flows[ft], dates_all, milton_geoids, milton_hrc_config,
        landing_date=landing_date,
    )
    milton_drop_dfs[ft] = drop_df
    milton_baselines[ft] = baselines
    milton_failures[ft] = failures

    print(f"Summary: {drop_df['largest_drop'].notna().sum()}/{len(drop_df)} valid")
    display(drop_df["largest_drop"].describe())


--- Milton: Largest drop for WITHIN ---
  Processing county 1/21: GEOID 12115

Completed: 21/21 counties with valid largest_drop
Summary: 21/21 valid


count    21.000000
mean    -35.033678
std       6.510959
min     -47.237604
25%     -40.776471
50%     -35.214422
75%     -30.990315
max     -23.527618
Name: largest_drop, dtype: float64


--- Milton: Largest drop for INFLOW ---
  Processing county 1/21: GEOID 12115


/Users/qing/miniconda3/envs/geo_env/lib/python3.11/site-packages/pandas/core/arraylike.py:399: RuntimeWarning: invalid value encountered in log1p
  result = getattr(ufunc, method)(*inputs, **kwargs)
/Users/qing/miniconda3/envs/geo_env/lib/python3.11/site-packages/pandas/core/arraylike.py:399: RuntimeWarning: invalid value encountered in log1p
  result = getattr(ufunc, method)(*inputs, **kwargs)


    SARIMA failed for GEOID 12081: A constant trend was included in the model specification, but the `exog` data already contains a column of constants.
    SARIMA failed for GEOID 12027: zero-size array to reduction operation maximum which has no identity


...

    SARIMA failed for GEOID 12097: zero-size array to reduction operation maximum which has no identity


/Users/qing/miniconda3/envs/geo_env/lib/python3.11/site-packages/pandas/core/arraylike.py:399: RuntimeWarning: invalid value encountered in log1p
  result = getattr(ufunc, method)(*inputs, **kwargs)
/Users/qing/miniconda3/envs/geo_env/lib/python3.11/site-packages/pandas/core/arraylike.py:399: RuntimeWarning: invalid value encountered in log1p
  result = getattr(ufunc, method)(*inputs, **kwargs)


    SARIMA failed for GEOID 12069: zero-size array to reduction operation maximum which has no identity
    SARIMA failed for GEOID 12053: zero-size array to reduction operation maximum which has no identity
    SARIMA failed for GEOID 12117: A constant trend was included in the model specification, but the `exog` data already contains a column of constants.
    SARIMA failed for GEOID 12127: A constant trend was included in the model specification, but the `exog` data already contains a column of constants.

Completed: 11/21 counties with valid largest_drop
Summary: 11/21 valid


/Users/qing/miniconda3/envs/geo_env/lib/python3.11/site-packages/pandas/core/arraylike.py:399: RuntimeWarning: invalid value encountered in log1p
  result = getattr(ufunc, method)(*inputs, **kwargs)
/Users/qing/miniconda3/envs/geo_env/lib/python3.11/site-packages/pandas/core/arraylike.py:399: RuntimeWarning: invalid value encountered in log1p
  result = getattr(ufunc, method)(*inputs, **kwargs)
/Users/qing/miniconda3/envs/geo_env/lib/python3.11/site-packages/pandas/core/arraylike.py:399: RuntimeWarning: invalid value encountered in log1p
  result = getattr(ufunc, method)(*inputs, **kwargs)


count      11.000000
mean     -410.053609
std      1190.006517
min     -3995.607476
25%       -68.232568
50%       -51.391211
75%       -31.054100
max         5.345929
Name: largest_drop, dtype: float64

### 2.2 Compute Outflow Increase

In [17]:
print("\n--- Milton: Outflow increase ---")
milton_outflow_df, milton_outflow_baselines, milton_outflow_failures = \
    compute_outflow_increase(
        milton_flows["outflow"], dates_all, milton_geoids, milton_hrc_config,
        landing_date=landing_date,
    )
print(f"Summary: {milton_outflow_df['largest_increase'].notna().sum()}/{len(milton_outflow_df)} valid")
display(milton_outflow_df["largest_increase"].describe())
display(milton_outflow_df)


--- Milton: Outflow increase ---
  Processing county 1/21: GEOID 12115

Completed: 21/21 counties with valid outflow increase
Summary: 21/21 valid


count     21.000000
mean      61.095320
std       51.381294
min        9.392404
25%       26.526502
50%       40.180513
75%       76.449953
max      177.825958
Name: largest_increase, dtype: float64

,GEOID,largest_increase,peak_date
0,12115,161.568177,2024-10-09
1,12081,165.568973,2024-10-08
2,12027,32.515765,2024-10-07
3,12015,177.825958,2024-10-08
4,12049,23.629464,2024-10-07
5,12057,94.568620,2024-10-09
6,12071,60.579897,2024-10-07
7,12103,76.449953,2024-10-09
8,12105,28.760332,2024-10-09
9,12055,11.122808,2024-10-14


### 2.3 Compute Recovery Time (within + inflow)

In [18]:
milton_recovery_dfs = {}
milton_recovery_results = {}

for ft in ["within", "inflow"]:
    print(f"\n--- Milton: Recovery time for {ft.upper()} ---")
    rec_df, rec_results = compute_county_recovery(
        milton_baselines[ft], milton_geoids, landing_date
    )
    milton_recovery_dfs[ft] = rec_df
    milton_recovery_results[ft] = rec_results

    display(rec_df["recovery_days"].describe())


--- Milton: Recovery time for WITHIN ---
  Recovery 1/21: GEOID 12115

Recovery completed: 21/21 counties with valid recovery time


count    21.000000
mean      4.868708
std       0.489750
min       4.143078
25%       4.553001
50%       4.801458
75%       5.098491
max       6.219785
Name: recovery_days, dtype: float64


--- Milton: Recovery time for INFLOW ---
  Recovery 1/21: GEOID 12115

Recovery completed: 9/21 counties with valid recovery time


count     9.000000
mean      5.799817
std       3.437048
min       1.703514
25%       4.593638
50%       4.931437
75%       6.389830
max      12.703205
Name: recovery_days, dtype: float64

### 2.4 Export All Baseline Validation Plots

In [19]:
# Export baseline plots for within + inflow + outflow
for ft in ["within", "inflow"]:
    export_all_baseline_plots(
        milton_baselines[ft], "milton", ft, landing_date, output_dir
    )

export_all_baseline_plots(
    milton_outflow_baselines, "milton", "outflow", landing_date, output_dir
)

Exporting 21 baseline plots for milton/within...
  Done: 21 plots saved to ../results/milton/figures/
Exporting 11 baseline plots for milton/inflow...
  Done: 11 plots saved to ../results/milton/figures/
Exporting 21 baseline plots for milton/outflow...
  Done: 21 plots saved to ../results/milton/figures/


### 2.5 Export All Recovery Plots

In [20]:
for ft in ["within", "inflow"]:
    print(f"\nExporting recovery plots for Milton/{ft}...")
    export_all_recovery_plots(
        milton_recovery_results[ft], milton_forecast_idx, milton_geoids,
        ft, "milton", landing_date, output_dir
    )


Exporting recovery plots for Milton/within...
  Exported 21 recovery plots for milton/within

Exporting recovery plots for Milton/inflow...
  Exported 11 recovery plots for milton/inflow


### 2.6 Export Milton CSVs

In [21]:
# Baselines + drop for within/inflow
for ft in ["within", "inflow"]:
    export_baselines(milton_baselines[ft], milton_drop_dfs[ft], output_dir, ft)

# Outflow baseline + increase
export_baselines(milton_outflow_baselines, milton_outflow_df, output_dir, "outflow")

# Recovery
for ft in ["within", "inflow"]:
    export_recovery(milton_recovery_dfs[ft], output_dir, ft)

Exported baseline to ../results/milton/baseline_within.csv (609 rows)
Exported to ../results/milton/largest_drop_within.csv (21 rows)
Exported baseline to ../results/milton/baseline_inflow.csv (319 rows)
Exported to ../results/milton/largest_drop_inflow.csv (21 rows)
Exported baseline to ../results/milton/baseline_outflow.csv (609 rows)
Exported to ../results/milton/outflow_increase.csv (21 rows)
Exported recovery to ../results/milton/recovery_within.csv (21 rows)
Exported recovery to ../results/milton/recovery_inflow.csv (21 rows)


---
## 3. Hurricane Helene

In [ ]:
# ── Load Helene county list ──
hrc_name = "helene"
hrc_cfg = HURRICANES[hrc_name]
landing_date = hrc_cfg["landing_date"]
output_dir = hrc_cfg["output_dir"]

with open(hrc_cfg["county_file"], "r") as f:
    helene_county_list = [int(line.strip()) for line in f if line.strip()]

helene_geo = geo_idx[geo_idx["GEOID"].isin(helene_county_list)].sort_values("county_idx")
helene_selected_idx = helene_geo["county_idx"].values
helene_geoids = helene_geo["GEOID"].tolist()

train_2024_end = (landing_date - pd.Timedelta(days=7)).strftime("%Y-%m-%d")
forecast_start = (landing_date - pd.Timedelta(days=6)).strftime("%Y-%m-%d")
forecast_end = "2024-10-31"

helene_hrc_config = {
    "train_2024_end": train_2024_end,
    "forecast_start": forecast_start,
    "forecast_end": forecast_end,
}
helene_forecast_idx = pd.date_range(forecast_start, forecast_end, freq="D")

print(f"Helene: {len(helene_geoids)} affected counties")
print(f"Landing date: {landing_date.date()}")
print(f"Train end: {train_2024_end}, Forecast: {forecast_start} -> {forecast_end}")

In [ ]:
# ── Load Helene mobility data ──
print("Loading Helene county-level flow data...")
helene_flows = load_county_flows(helene_selected_idx)

### 3.1 Compute Largest Drop (within + inflow)

In [ ]:
helene_drop_dfs = {}
helene_baselines = {}
helene_failures = {}

for ft in ["within", "inflow"]:
    print(f"\n--- Helene: Largest drop for {ft.upper()} ---")
    drop_df, baselines, failures = compute_baseline_and_drop(
        helene_flows[ft], dates_all, helene_geoids, helene_hrc_config,
        landing_date=HURRICANES["helene"]["landing_date"],
    )
    helene_drop_dfs[ft] = drop_df
    helene_baselines[ft] = baselines
    helene_failures[ft] = failures

    print(f"Summary: {drop_df['largest_drop'].notna().sum()}/{len(drop_df)} valid")
    display(drop_df["largest_drop"].describe())

### 3.2 Compute Outflow Increase

In [ ]:
print("\n--- Helene: Outflow increase ---")
helene_outflow_df, helene_outflow_baselines, helene_outflow_failures = \
    compute_outflow_increase(
        helene_flows["outflow"], dates_all, helene_geoids, helene_hrc_config,
        landing_date=HURRICANES["helene"]["landing_date"],
    )
print(f"Summary: {helene_outflow_df['largest_increase'].notna().sum()}/{len(helene_outflow_df)} valid")
display(helene_outflow_df["largest_increase"].describe())

### 3.3 Compute Recovery Time (within + inflow)

In [ ]:
helene_recovery_dfs = {}
helene_recovery_results = {}

for ft in ["within", "inflow"]:
    print(f"\n--- Helene: Recovery time for {ft.upper()} ---")
    rec_df, rec_results = compute_county_recovery(
        helene_baselines[ft], helene_geoids,
        HURRICANES["helene"]["landing_date"]
    )
    helene_recovery_dfs[ft] = rec_df
    helene_recovery_results[ft] = rec_results

    display(rec_df["recovery_days"].describe())

### 3.4 Export All Baseline Validation Plots

In [ ]:
helene_landing = HURRICANES["helene"]["landing_date"]
helene_output = HURRICANES["helene"]["output_dir"]

for ft in ["within", "inflow"]:
    export_all_baseline_plots(
        helene_baselines[ft], "helene", ft, helene_landing, helene_output
    )

export_all_baseline_plots(
    helene_outflow_baselines, "helene", "outflow", helene_landing, helene_output
)

### 3.5 Export All Recovery Plots

In [ ]:
for ft in ["within", "inflow"]:
    print(f"\nExporting recovery plots for Helene/{ft}...")
    export_all_recovery_plots(
        helene_recovery_results[ft], helene_forecast_idx, helene_geoids,
        ft, "helene", helene_landing, helene_output
    )

### 3.6 Export Helene CSVs

In [ ]:
# Baselines + drop for within/inflow
for ft in ["within", "inflow"]:
    export_baselines(helene_baselines[ft], helene_drop_dfs[ft], helene_output, ft)

# Outflow baseline + increase
export_baselines(helene_outflow_baselines, helene_outflow_df, helene_output, "outflow")

# Recovery
for ft in ["within", "inflow"]:
    export_recovery(helene_recovery_dfs[ft], helene_output, ft)

---
## 4. Summary of Exported Files

In [ ]:
print("=" * 70)
print("EXPORTED FILES SUMMARY")
print("=" * 70)

expected_csvs = [
    "baseline_within.csv", "baseline_inflow.csv", "baseline_outflow.csv",
    "largest_drop_within.csv", "largest_drop_inflow.csv",
    "outflow_increase.csv",
    "recovery_within.csv", "recovery_inflow.csv",
]

for hrc_name, cfg in HURRICANES.items():
    out_dir = cfg["output_dir"]
    print(f"\n--- {hrc_name.upper()} ({out_dir}) ---")
    for fname in expected_csvs:
        p = os.path.join(out_dir, fname)
        if os.path.exists(p):
            df_tmp = pd.read_csv(p)
            print(f"  {fname:35s} {len(df_tmp):>6d} rows")
        else:
            print(f"  {fname:35s} NOT FOUND")
    fig_dir = os.path.join(out_dir, "figures")
    if os.path.isdir(fig_dir):
        figs = [f for f in os.listdir(fig_dir) if f.endswith(".png")]
        print(f"  figures/:  {len(figs)} PNG files")

print("\nNotebook 1 complete. Proceed to Notebook 2 (v2b) for regression analysis.")